# 28B Heartbeat: within-filing signals, detect, decide, deduplicate
Every signal compares this period's schedule with the prior fiscal year-end schedule **in the same filing**. Pairing the two is the hard part, and it is yours to design.

In [1]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

fixture mode: True | clock: 2026-10-15T06:00:00-04:00


In [2]:
heads = json.load(open(FX / "release_heads.json"))
tick = heartbeat.release_tick(heads["last_seen"], lambda url: heads["heads"].get(url, 404), P["sec"]["base_url"])
print(tick)

{'release': '2026_09', 'url': 'https://www.sec.gov/files/datastandardsinnovation/data/business-development-company-bdc-data-sets/2026_09_bdc.zip', 'published': True}


In [3]:
pairs = signals.pair_exact(PANEL)
print("Exact pairing (same filing, identical identifier):")
print(signals.pairing_report(PANEL, pairs).to_string(index=False))
ex = PANEL[PANEL["name"].str.contains("GOLF") & PANEL["identifier"].str.contains("Cedar Ridge")]
print("\nWhy Golf pairs nothing: the identifier carries the all-in rate, which moved with SOFR")
print(ex[["period_role", "identifier"]].to_string(index=False))

Exact pairing (same filing, identical identifier):
                adsh                         name  debt_holdings  paired  unpaired
0009000011-26-000111    FIXTURE GOLF CAPITAL CORP             10       0        10
0009000012-26-000121 FIXTURE HOTEL SENIOR LENDING              8       8         0
0009000013-26-000131    FIXTURE INDIA CREDIT FUND              7       6         1
0009000014-26-000141  FIXTURE JULIETT LENDING BDC             10       9         1

Why Golf pairs nothing: the identifier carries the all-in rate, which moved with SOFR
period_role                                                                                      identifier
      prior First Lien Senior Secured Loan | Cedar Ridge Dental Partners LLC | S + 5.75% (10.05%) | 09/2030
    current  First Lien Senior Secured Loan | Cedar Ridge Dental Partners LLC | S + 5.75% (9.80%) | 09/2030


In [4]:
found = signals.find_signals(PANEL, M)
print(found[["name", "rule", "identifier", "row", "before", "after"]].to_string(index=False))
if P["data"]["use_fixture"]:
    truth = pd.read_csv(ROOT / P["data"]["truth"], dtype=str)
    sc = evals.signal_scores(found, truth)
    print(f"\nAgainst the planted events: precision {sc['precision']:.2f}, recall {sc['recall']:.2f} "
          f"({sc['tp']} found, {sc['fp']} false, {sc['fn']} missed)")
    print(pd.DataFrame(sc["by_rule"]).T)

                        name     rule                                                        identifier row     before      after
FIXTURE HOTEL SENIOR LENDING EXTENDED               Pinecrest Learning Group Inc., First Lien Term Loan 105 2029-09-30 2031-09-30
FIXTURE HOTEL SENIOR LENDING MARKDOWN            Brightwater Fitness Holdings LLC, First Lien Term Loan 101       99.0       93.5
FIXTURE HOTEL SENIOR LENDING   PIK_UP                   Oakmont Home Services LLC, First Lien Term Loan 104        1.0        3.0
   FIXTURE INDIA CREDIT FUND MARKDOWN Senior Secured First Lien | Westbrook Orthopedics LLC | S + 5.50% 102       96.0       89.5
 FIXTURE JULIETT LENDING BDC MARKDOWN   First Lien Term Loan | Crestline Roofing Holdings LLC | 06/2031 103       97.5       90.0
 FIXTURE JULIETT LENDING BDC   PIK_UP   First Lien Term Loan | Foxglove Specialty Pharma Inc. | 03/2031 106        0.0        2.5

Against the planted events: precision 1.00, recall 0.67 (6 found, 0 false, 3 missed)
    

In [5]:
problems = panel.check_mandates(MANDATES, PANEL)    # a misspelled BDC name would route nothing, silently
assert not problems, problems
held = {m: list(v["positions"]) for m, v in MANDATES.items()}
routed = heartbeat.route(found, held)
seen = set()
early = [a for m, s in routed.items() for a in heartbeat.decide(s, m, seen, NOW)]
print(f"At {NOW:%H:%M} (quiet hours):", pd.Series([a["action"] for a in early]).value_counts().to_dict(), "| per mandate:",
      pd.Series([a["mandate"] for a in early]).value_counts().to_dict())
again = [a for m, s in routed.items() for a in heartbeat.decide(s, m, seen, NOW.replace(hour=9))]
print("Same signals at 09:00:", again, "(deduplicated)")
recent = json.load(open(FX / "edgar_recent.json"))
for bdc, cik in REG["bdc_cik"].items():
    new = heartbeat.new_filings(recent.get(cik, {"form": [], "items": [], "accessionNumber": [], "acceptanceDateTime": []}), seen=set())
    for m in MANDATES:
        for d in heartbeat.filing_decisions(new, bdc, held[m], m, seen, NOW.replace(hour=9)):
            print(f"  {m}: {d['form']:5s} {d['accn']} {bdc.replace('FIXTURE ', ''):24s} -> {d['action']}")

At 06:00 (quiet hours): {'hold': 9} | per mandate: {'mandate_a': 5, 'mandate_b': 4}
Same signals at 09:00: [] (deduplicated)
  mandate_a: 10-Q  0009000011-26-000111 GOLF CAPITAL CORP        -> alert
  mandate_a: 8-K   0009000011-26-000109 GOLF CAPITAL CORP        -> alert
  mandate_b: 10-Q  0009000011-26-000111 GOLF CAPITAL CORP        -> not_material
  mandate_b: 8-K   0009000011-26-000109 GOLF CAPITAL CORP        -> not_material
  mandate_a: 10-Q  0009000013-26-000131 INDIA CREDIT FUND        -> not_material
  mandate_a: 8-K   0009000013-26-000130 INDIA CREDIT FUND        -> not_material
  mandate_b: 10-Q  0009000013-26-000131 INDIA CREDIT FUND        -> alert
  mandate_b: 8-K   0009000013-26-000130 INDIA CREDIT FUND        -> alert


Exact pairing finds 6 of the 9 planted events with no false alarms: precision 1.00, recall 0.67. It misses Golf's two markdowns (the all-in rate in the identifier changed) and Juliett's extension (the maturity in the identifier changed). Two traps punish a careless fix: Golf's Granite Peak has a term loan and a delayed-draw tranche, only one of them marked down, and Hotel's Redwood Valley fell 4.5 points, under the 5-point threshold.

**Your turn.** Write your own pairing function, pass it as `signals.find_signals(panel, m, pair=your_pairing)`, and measure it: precision and recall on the fixture, plus a hand-checked sample of pairs from your real panel. Then build the four DAGs, deliver alerts and the brief, and measure latency, alerts per mandate-week and idle cost (Part 5).